In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# 동적 Circuit 실행

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="패키지 버전">

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
동적 Circuit은 양자 Circuit 실행 중간에 Qubit을 측정하고, 해당 중간 회로 측정 결과를 기반으로 Circuit 내에서 고전 논리 연산을 수행할 수 있는 강력한 도구입니다. 이 프로세스는 _고전적 피드포워드_라고도 합니다. 동적 Circuit을 최대한 활용하는 방법에 대한 이해는 아직 초기 단계이지만, 양자 연구 커뮤니티는 이미 다음과 같은 여러 사용 사례를 확인했습니다:

* [GHZ 상태](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339), [W-상태](https://arxiv.org/abs/2403.07604)(W-상태에 대한 자세한 내용은 ["피드포워드를 사용한 얕은 Circuit에 의한 상태 준비"](https://arxiv.org/abs/2307.14840)도 참조)와 같은 효율적인 양자 상태 준비, 그리고 광범위한 [행렬 곱 상태](https://arxiv.org/abs/2404.16083)
* 얕은 Circuit을 사용하여 동일한 칩의 Qubit 간 효율적인 [장거리 얽힘](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339)
* [IQP 유사 Circuit의 효율적인 샘플링](https://arxiv.org/pdf/2505.04705)

그러나 동적 Circuit으로 인한 이러한 개선에는 트레이드오프가 따릅니다. 중간 회로 측정과 고전 연산은 일반적으로 2-Qubit 게이트보다 실행 시간이 더 길며, 이 시간 증가가 회로 깊이 감소의 이점을 상쇄할 수 있습니다. 따라서 중간 회로 측정 길이 단축은 IBM Quantum&reg;이 동적 Circuit의 [새 버전](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits)을 출시함에 따라 개선의 핵심 영역입니다. 동적 Circuit 사용 시의 다른 제한 사항은 [Estimator](/guides/estimator-options#options-compatibility-table) 또는 [Sampler](/guides/sampler-options#options-compatibility-table) 기능 호환성 표를 참조하세요.

[OpenQASM 3 명세](https://openqasm.com/language/classical.html#looping-and-branching)는 다수의 제어 흐름 구조를 정의하지만, Qiskit Runtime은 현재 조건부 `if` 문만 지원합니다. Qiskit SDK에서 이는 [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit)의 [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) 메서드에 해당합니다. 이 메서드는 [컨텍스트 관리자](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers)를 반환하며 일반적으로 `with` 문에서 사용됩니다. 이 가이드에서는 이 조건문 사용 방법을 설명합니다.

> **Note:** 이 가이드의 코드 예제는 중간 회로 측정에 표준 measure 명령을 사용합니다. 그러나 Backend가 지원하는 경우 `MidCircuitMeasure` 명령을 대신 사용하는 것이 권장됩니다. 자세한 내용은 [중간 회로 측정 섹션](#midcircuit)을 참조하세요.
## 동적 Circuit을 지원하는 Backend 찾기
계정이 접근할 수 있고 동적 Circuit을 지원하는 모든 Backend를 찾으려면 다음과 같은 코드를 실행하세요. 이 예제는 [로그인 자격 증명을 저장](/guides/save-credentials)했다고 가정합니다. Qiskit Runtime 서비스 계정을 초기화할 때 [명시적으로 자격 증명을 지정](/guides/initialize-account#explicit)할 수도 있습니다. 이를 통해 특정 인스턴스나 플랜 유형에서 사용 가능한 Backend를 볼 수 있습니다.

> **Note:** - 계정에서 사용 가능한 Backend는 자격 증명에서 지정된 인스턴스에 따라 다릅니다.
> - 동적 Circuit의 새 버전은 이제 모든 Backend의 모든 사용자에게 제공됩니다. 자세한 내용은 [공지 사항](/announcements/product-updates/2025-09-25-new-dynamic-circuits)을 참조하세요.

In [1]:
# This cell is hidden from users.  It hides all those "...instance was not set..." warnings.
import warnings

warnings.filterwarnings("ignore", message=".*Instance was not set*")

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
dc_backends = service.backends(dynamic_circuits=True)
print(dc_backends)

[<IBMBackend('ibm_boston')>, <IBMBackend('ibm_pittsburgh')>, <IBMBackend('ibm_fez')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_kingston')>]


<span id="midcircuit"></span>
## Mid-circuit measurements

Prior to `qiskit-ibm-runtime` v0.43.0, `measure` was the only measurement instruction in Qiskit. Mid-circuit measurements, however, have different tuning requirements than _terminal_ measurements (measurements that happen at the end of a circuit). For example, you need to consider the instruction duration when tuning a mid-circuit measurement because longer instructions cause noisier circuits. You don't need to consider instruction duration for terminal measurements because there are no instructions after terminal measurements.

<Admonition type="note">
The `MidCircuitMeasure` instruction maps to the `measure_2` instruction reported in the backend's `supported_instructions`. However,  `measure_2` is not supported on all backends. Use `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` to find backends that support it.  New measurements might be added in the future, but this is not guarenteed.
</Admonition>

### `MidCircuitMeasure` method

In `qiskit-ibm-runtime` v0.43.0, the `MidCircuitMeasure` instruction was introduced. As the name suggests, it is a new measurement instruction that is optimized for mid-circuit on IBM&reg; QPUs.  While you can use `QuantumCircuit.measure` for a mid-circuit measurement, because of its design, `MidCircuitMeasure` is typically a better choice.  For example, it adds less overhead to your circuit than when using `QuantumCircuit.measure`.

In [3]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.circuit import MidCircuitMeasure

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, dynamic_circuits=True
)

circ = QuantumCircuit(2, 2)
circ.x(0)
circ.append(MidCircuitMeasure(), [0], [0])
# circ.measure([0], [0])
# circ.measure_all()
print(circ.draw(cregbundle=False))

     ┌───┐┌────────────┐
q_0: ┤ X ├┤0           ├
     └───┘│            │
q_1: ─────┤  Measure_2 ├
          │            │
c_0: ═════╡0           ╞
          └────────────┘
c_1: ═══════════════════
                        


<span id="midcircuit"></span>
## 중간 회로 측정
`qiskit-ibm-runtime` v0.43.0 이전에는 `measure`가 Qiskit의 유일한 측정 명령이었습니다. 그러나 중간 회로 측정은 _터미널_ 측정(Circuit 끝에서 수행되는 측정)과는 다른 튜닝 요구 사항을 가집니다. 예를 들어, 중간 회로 측정을 튜닝할 때는 명령 지속 시간을 고려해야 하는데, 더 긴 명령은 더 노이즈가 많은 Circuit을 만들기 때문입니다. 터미널 측정 후에는 명령이 없으므로 터미널 측정에서는 명령 지속 시간을 고려할 필요가 없습니다.

> **Note:** `MidCircuitMeasure` 명령은 Backend의 `supported_instructions`에 보고된 `measure_2` 명령에 매핑됩니다. 그러나 `measure_2`는 모든 Backend에서 지원되지는 않습니다. `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)`를 사용하여 이를 지원하는 Backend를 찾으세요. 향후 새 측정 명령이 추가될 수 있지만, 이는 보장되지 않습니다.

### `MidCircuitMeasure` 메서드
`qiskit-ibm-runtime` v0.43.0에서 `MidCircuitMeasure` 명령이 도입되었습니다. 이름에서 알 수 있듯이, IBM&reg; QPU에서 중간 회로에 최적화된 새로운 측정 명령입니다. 중간 회로 측정에 `QuantumCircuit.measure`를 사용할 수 있지만, 설계 특성상 `MidCircuitMeasure`가 일반적으로 더 나은 선택입니다. 예를 들어, `QuantumCircuit.measure`를 사용할 때보다 Circuit에 더 적은 오버헤드를 추가합니다.

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, dynamic_circuits=True
)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)


# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d82864vtjchs73bnokf0 (DONE)


## Qiskit Runtime limitations

Be aware of the following constraints when running dynamic circuits in Qiskit Runtime.

- Due to the limited physical memory on control electronics, there is also a limit on the number of `if` statements and size of their operands. This limit is a function of the number of broadcasts and the number of broadcasted bits in a job (not a circuit).

   When processing an `if` condition, measurement data needs to be transferred to the control logic to make that evaluation. A broadcast is a transfer of unique classical data, and broadcasted bits is the number of classical bits being transferred. Consider the following:

   ```python
   c0 = ClassicalRegister(3)
   c1 = ClassicalRegister(5)
   ...
   with circuit.if_test((c0, 1)) ...
   with circuit.if_test((c0, 3)) ...
   with circuit.if_test((c1[2], 1)) ...
   ```
   In the previous code example, the first two `if_test` objects on `c0` are considered one broadcast because the content of `c0` did not change, and thus does not need to be re-broadcasted. The `if_test` on `c1` is a second broadcast. The first one broadcasts all three bits in `c0` and the second broadcasts just one bit, making a total of four broadcasted bits.

   Currently, if you broadcast 60 bits each time, then the job can have approximately 300 broadcasts. If you broadcast just one bit each time, however, then the job can have 2400 broadcasts.

- The operand used in an `if_test` statement must be 32 or fewer bits. Thus, if you are comparing an entire `ClassicalRegister`, the size of that `ClassicalRegister` must be 32 or fewer bits. If you are comparing just a single bit from a `ClassicalRegister`, however, that `ClassicalRegister` can be of any size (since the operand is only one bit).

   For example, the "Not valid" code block does not work because `cr` is more than 32 bits.  You can, however, use a classical register wider than 32 bits if you are testing only one bit, as shown in the "Valid" code block.

   <Tabs>
   <TabItem value="Not valid" label="Not valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr, 15)):
            ...
      ```
   </TabItem>
   <TabItem value="Valid" label="Valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr[5], 1)):
            ...
      ```
   </TabItem>
   </Tabs>

- Nested conditionals are not allowed. For example, the following code block will not work because it has an `if_test` inside another `if_test`:
   <Tabs>
    <TabItem value="Not valid" label="Not valid">
        ```python
           c1 = ClassicalRegister(1, "c1")
           c2 = ClassicalRegister(2, "c2")
           ...
           with circ.if_test((c1, 1)):
            with circ.if_test(c2, 1)):
             ...
        ```
     </TabItem>
     <TabItem value="Valid" label="Valid">
        ```python
        cr = ClassicalRegister(2)
        ...
        with circuit.if_test((cr, 0b11)):
          ...
        ```
     </TabItem>
    </Tabs>

- Having `reset` or measurements inside conditionals is not supported.
- Arithmetic operations are not supported.
- See the [OpenQASM 3 feature table](/docs/guides/qasm-feature-table) to determine which OpenQASM 3 features are supported on Qiskit and Qiskit Runtime.
- When OpenQASM 3 (instead of `QuantumCircuit`) is used as the input format to pass circuits to Qiskit Runtime primitives, only instructions that can be loaded into Qiskit are supported. Classical operations, for example, are not supported because they cannot be loaded into Qiskit. See [Import an OpenQASM 3 program into Qiskit](/docs/guides/interoperate-qiskit-qasm3#import-an-openqasm-3-program-into-qiskit) for more information.
- The `for`, `while`, and `switch` instructions are not supported.

## Use dynamic circuits with Estimator

Since Estimator does not support dynamic circuits, you can use [Sampler](/docs/guides/get-started-with-sampler) and build your own measurement circuits instead.

To replicate Estimator's behavior, follow this process:

1. Group the terms of all observables into a partition.  This can be done by using the [`PauliList` API](/docs/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting), for example.
     <Admonition type="note">
      You can use the [`BitArray`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.BitArray#expectation_values) primitive attribute to compute the expectation values of the provided observables.
     </Admonition>
2. Execute one basis change circuit per partition (whichever basis change needs to be done for each partition). See the Measurement bases addon utility [`measurement_bases` module](https://qiskit.github.io/qiskit-addon-utils/apidocs/qiskit_addon_utils.exp_vals.html#qiskit_addon_utils.exp_vals.get_measurement_bases) for more information. For more information, see the [documentation](https://qiskit.github.io/qiskit-addon-utils/) for the Qiskit addon utilities package.
3. Add back together the results for each partition.

## Restrictions

Review any [Feature compatibility table](/docs/guides/estimator-options#options-compatibility-table) to understand restrictions when using dynamic circuits. Note that feature compatibility is not primitive-dependent.

## Next steps

<Admonition type="tip" title="Recommendations">
- Learn how to implement accurate dynamic decoupling by using [stretch](/docs/guides/stretch).
- Review the [classical feedforward and control flow](/docs/guides/classical-feedforward-and-control-flow) guide.
- Use [circuit schedule visualization](/docs/guides/qiskit-runtime-circuit-timing) to debug and optimize your dynamic circuits.
</Admonition>